# FACT ORDERS

In [0]:
from pyspark.sql import functions as F

orders = spark.table("lakehouse.silver.orders")
customers = spark.table("lakehouse.silver.customers")

dim_customers = spark.table("lakehouse.gold.dim_customers")
dim_geography = spark.table("lakehouse.gold.dim_geography")
dim_date = spark.table("lakehouse.gold.dim_date")

# Check order grain
orders.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("distinct_orders"),
    F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0))
        .alias("null_order_ids")
).show()

# Check customer lookup key
customers.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("distinct_customer_ids"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0))
        .alias("null_customer_ids")
).show()

# Check customer dimension key
dim_customers.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_unique_id").alias("distinct_customer_ids")
).show()

# Check dimension availability and schemas
print("Geography dimension:")
dim_geography.printSchema()

print("Date dimension:")
dim_date.printSchema()

print("Date dimension rows:", dim_date.count())

In [0]:
customers = spark.table("lakehouse.silver.customers")
dim_customers = spark.table("lakehouse.gold.dim_customers")
dim_geography = spark.table("lakehouse.gold.dim_geography")

customer_lookup = (
    customers.alias("c")
    .join(
        dim_customers.alias("dc"),
        F.col("c.customer_unique_id") == F.col("dc.customer_unique_id"),
        "left"
    )
    .join(
        dim_geography.alias("dg"),
        ( 
            (F.col("c.customer_zip_code_prefix") == F.col("dg.zip_code_prefix")) &
            (F.col("c.customer_city") == F.col("dg.city")) &
            (F.col("c.customer_state") == F.col("dg.state"))
        ),
        "left"
    )
    .select(
        F.col("c.customer_id"),
        F.col("dc.customer_sk"),
        F.col("dg.geography_sk").alias("customer_geography_sk")
    )
)

In [0]:
customer_lookup.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("distinct_customer_ids"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0))
        .alias("null_customer_ids"),
    F.sum(F.when(F.col("customer_sk").isNull(), 1).otherwise(0))
        .alias("missing_customer_keys"),
    F.sum(F.when(F.col("customer_geography_sk").isNull(), 1).otherwise(0))
        .alias("missing_geography_keys")
).show()

In [0]:
orders = spark.table("lakehouse.silver.orders")
dim_date = spark.table("lakehouse.gold.dim_date")

In [0]:
orders_with_dates = (
    orders.alias("o")

    .join(
        dim_date.alias("purchase"),
        F.to_date(F.col("o.order_purchase_timestamp")) == F.col("purchase.date"),
        "left"
    )

    .join(
        dim_date.alias("approval"),
        F.to_date(F.col("o.order_approved_at")) == F.col("approval.date"),
        "left"
    )

    .join(
        dim_date.alias("carrier"),
        F.to_date(F.col("o.order_delivered_carrier_date")) == F.col("carrier.date"),
        "left"
    )

    .join(
        dim_date.alias("delivery"),
        F.to_date(F.col("o.order_delivered_customer_date")) == F.col("delivery.date"),
        "left"
    )

    .join(
        dim_date.alias("estimated"),
        F.to_date(F.col("o.order_estimated_delivery_date")) == F.col("estimated.date"),
        "left"
    )

    .select(
        F.col("o.order_id"),
        F.col("purchase.date_sk").alias("purchase_date_sk"),
        F.col("approval.date_sk").alias("approval_date_sk"),
        F.col("carrier.date_sk").alias("carrier_date_sk"),
        F.col("delivery.date_sk").alias("delivery_date_sk"),
        F.col("estimated.date_sk").alias("estimated_delivery_date_sk")
    )
)

In [0]:
orders_with_dates.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("distinct_orders"),

    F.sum(F.when(F.col("purchase_date_sk").isNull(), 1).otherwise(0))
        .alias("missing_purchase_date_keys"),

    F.sum(F.when(F.col("approval_date_sk").isNull(), 1).otherwise(0))
        .alias("missing_approval_date_keys"),

    F.sum(F.when(F.col("carrier_date_sk").isNull(), 1).otherwise(0))
        .alias("missing_carrier_date_keys"),

    F.sum(F.when(F.col("delivery_date_sk").isNull(), 1).otherwise(0))
        .alias("missing_delivery_date_keys"),

    F.sum(F.when(F.col("estimated_delivery_date_sk").isNull(), 1).otherwise(0))
        .alias("missing_estimated_delivery_date_keys")
).show()

In [0]:
date_validation = orders_with_dates.alias("f").join(
    orders.alias("o"),
    F.col("f.order_id") == F.col("o.order_id"),
    "inner"
)

date_validation.select(
    F.count("*").alias("total_orders"),

    F.sum(
        F.when(F.col("o.order_approved_at").isNull(), 1).otherwise(0)
    ).alias("null_approval_timestamps"),

    F.sum(
        F.when(F.col("f.approval_date_sk").isNull(), 1).otherwise(0)
    ).alias("null_approval_date_keys"),

    F.sum(
        F.when(F.col("o.order_delivered_carrier_date").isNull(), 1).otherwise(0)
    ).alias("null_carrier_timestamps"),

    F.sum(
        F.when(F.col("f.carrier_date_sk").isNull(), 1).otherwise(0)
    ).alias("null_carrier_date_keys"),

    F.sum(
        F.when(F.col("o.order_delivered_customer_date").isNull(), 1).otherwise(0)
    ).alias("null_delivery_timestamps"),

    F.sum(
        F.when(F.col("f.delivery_date_sk").isNull(), 1).otherwise(0)
    ).alias("null_delivery_date_keys")
).show()

In [0]:
fact_orders = (
    orders.alias("o")

    .join(
        customer_lookup.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )

    .join(
        orders_with_dates.alias("d"),
        F.col("o.order_id") == F.col("d.order_id"),
        "left"
    )

    .select(
        # Business key
        F.col("o.order_id"),

        # Customer identifiers and dimension keys
        F.col("o.customer_id"),
        F.col("c.customer_sk"),
        F.col("c.customer_geography_sk"),

        # Order attributes
        F.col("o.order_status"),

        # Date dimension foreign keys
        F.col("d.purchase_date_sk"),
        F.col("d.approval_date_sk"),
        F.col("d.carrier_date_sk"),
        F.col("d.delivery_date_sk"),
        F.col("d.estimated_delivery_date_sk"),

        # Original timestamps
        F.col("o.order_purchase_timestamp"),
        F.col("o.order_approved_at"),
        F.col("o.order_delivered_carrier_date"),
        F.col("o.order_delivered_customer_date"),
        F.col("o.order_estimated_delivery_date")
    )
)

In [0]:
fact_orders.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("distinct_orders"),

    F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0))
        .alias("null_order_ids"),

    F.sum(F.when(F.col("customer_sk").isNull(), 1).otherwise(0))
        .alias("missing_customer_keys"),

    F.sum(F.when(F.col("customer_geography_sk").isNull(), 1).otherwise(0))
        .alias("missing_geography_keys"),

    F.sum(F.when(F.col("purchase_date_sk").isNull(), 1).otherwise(0))
        .alias("missing_purchase_date_keys")
).show()

In [0]:
def count_orphan_keys(fact_df, fact_key, dimension_df, dimension_key):

    fact_alias = fact_df.alias("f")
    dimension_alias = (
        dimension_df
        .select(F.col(dimension_key).alias("_dimension_key"))
        .distinct()
        .alias("d")
    )

    return (
        fact_alias
        .filter(F.col(f"f.{fact_key}").isNotNull())
        .join(
            dimension_alias,
            F.col(f"f.{fact_key}") == F.col("d._dimension_key"),
            "left_anti"
        )
        .count()
    )

In [0]:
dim_customers = spark.table("lakehouse.gold.dim_customers")
dim_geography = spark.table("lakehouse.gold.dim_geography")
dim_date = spark.table("lakehouse.gold.dim_date")

foreign_key_checks = {
    "customer_sk": count_orphan_keys(
        fact_orders, "customer_sk",
        dim_customers, "customer_sk"
    ),

    "customer_geography_sk": count_orphan_keys(
        fact_orders, "customer_geography_sk",
        dim_geography, "geography_sk"
    ),

    "purchase_date_sk": count_orphan_keys(
        fact_orders, "purchase_date_sk",
        dim_date, "date_sk"
    ),

    "approval_date_sk": count_orphan_keys(
        fact_orders, "approval_date_sk",
        dim_date, "date_sk"
    ),

    "carrier_date_sk": count_orphan_keys(
        fact_orders, "carrier_date_sk",
        dim_date, "date_sk"
    ),

    "delivery_date_sk": count_orphan_keys(
        fact_orders, "delivery_date_sk",
        dim_date, "date_sk"
    ),

    "estimated_delivery_date_sk": count_orphan_keys(
        fact_orders, "estimated_delivery_date_sk",
        dim_date, "date_sk"
    )
}

for key, count in foreign_key_checks.items():
    print(f"{key}: {count} orphan records")

In [0]:
fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.fact_orders")

# FACT ORDERS ITEMS

In [0]:
from pyspark.sql import functions as F

order_items = spark.table("lakehouse.silver.order_items")

dim_products = spark.table("lakehouse.gold.dim_products")
dim_sellers = spark.table("lakehouse.gold.dim_sellers")

order_items.printSchema()
order_items.show(10, truncate=False)

In [0]:
order_items.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct(
        F.struct("order_id", "order_item_id")
    ).alias("distinct_order_items"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("null_order_ids"),

    F.sum(
        F.when(F.col("order_item_id").isNull(), 1).otherwise(0)
    ).alias("null_order_item_ids"),

    F.sum(
        F.when(F.col("product_id").isNull(), 1).otherwise(0)
    ).alias("null_product_ids"),

    F.sum(
        F.when(F.col("seller_id").isNull(), 1).otherwise(0)
    ).alias("null_seller_ids")
).show()

In [0]:
print("PRODUCT DIMENSION")

dim_products.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.sum(
        F.when(F.col("product_id").isNull(), 1).otherwise(0)
    ).alias("null_product_ids"),
    F.sum(
        F.when(F.col("product_sk").isNull(), 1).otherwise(0)
    ).alias("null_product_keys")
).display()


print("SELLER DIMENSION")

dim_sellers.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("seller_id").alias("distinct_sellers"),
    F.sum(
        F.when(F.col("seller_id").isNull(), 1).otherwise(0)
    ).alias("null_seller_ids"),
    F.sum(
        F.when(F.col("seller_sk").isNull(), 1).otherwise(0)
    ).alias("null_seller_keys")
).display()

In [0]:
product_ids_in_source = (
    order_items
    .select("product_id")
    .distinct()
)

seller_ids_in_source = (
    order_items
    .select("seller_id")
    .distinct()
)

missing_products = (
    product_ids_in_source
    .join(
        dim_products.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

missing_sellers = (
    seller_ids_in_source
    .join(
        dim_sellers.select("seller_id"),
        on="seller_id",
        how="left_anti"
    )
)

print("Products missing from dimension:", missing_products.count())
print("Sellers missing from dimension:", missing_sellers.count())

In [0]:
order_items_with_keys = (
    order_items.alias("oi")

    .join(
        dim_products.alias("p"),
        F.col("oi.product_id") == F.col("p.product_id"),
        "left"
    )

    .join(
        dim_sellers.alias("s"),
        F.col("oi.seller_id") == F.col("s.seller_id"),
        "left"
    )

    .select(
        F.col("oi.order_id"),
        F.col("oi.order_item_id"),

        F.col("oi.product_id"),
        F.col("p.product_sk"),

        F.col("oi.seller_id"),
        F.col("s.seller_sk")
    )
)

In [0]:
order_items_with_keys.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct(
        F.struct("order_id", "order_item_id")
    ).alias("distinct_order_items"),

    F.sum(
        F.when(F.col("product_sk").isNull(), 1).otherwise(0)
    ).alias("missing_product_keys"),

    F.sum(
        F.when(F.col("seller_sk").isNull(), 1).otherwise(0)
    ).alias("missing_seller_keys")
).show()

In [0]:
order_items_with_keys.show(10, truncate=False)

In [0]:
fact_order_items = (
    order_items.alias("oi")

    .join(
        dim_products.alias("p"),
        F.col("oi.product_id") == F.col("p.product_id"),
        "left"
    )

    .join(
        dim_sellers.alias("s"),
        F.col("oi.seller_id") == F.col("s.seller_id"),
        "left"
    )

    .select(
        F.col("oi.order_id"),
        F.col("oi.order_item_id"),

        F.col("oi.product_id"),
        F.col("p.product_sk"),

        F.col("oi.seller_id"),
        F.col("s.seller_sk"),

        F.col("oi.shipping_limit_date"),
        F.col("oi.price"),
        F.col("oi.freight_value")
    )
)

In [0]:
fact_order_items.printSchema()

fact_order_items.show(10, truncate=False)

In [0]:
fact_order_items.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct(
        F.struct("order_id", "order_item_id")
    ).alias("distinct_order_items"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("null_order_ids"),

    F.sum(
        F.when(F.col("order_item_id").isNull(), 1).otherwise(0)
    ).alias("null_order_item_ids"),

    F.sum(
        F.when(F.col("product_sk").isNull(), 1).otherwise(0)
    ).alias("missing_product_keys"),

    F.sum(
        F.when(F.col("seller_sk").isNull(), 1).otherwise(0)
    ).alias("missing_seller_keys"),

    F.sum(
        F.when(F.col("price").isNull(), 1).otherwise(0)
    ).alias("null_prices"),

    F.sum(
        F.when(F.col("freight_value").isNull(), 1).otherwise(0)
    ).alias("null_freight_values")
).show()

In [0]:
item_fk_checks = {
    "product_sk": count_orphan_keys(
        fact_order_items,
        "product_sk",
        dim_products,
        "product_sk"
    ),

    "seller_sk": count_orphan_keys(
        fact_order_items,
        "seller_sk",
        dim_sellers,
        "seller_sk"
    )
}

for key, count in item_fk_checks.items():
    print(f"{key}: {count} orphan records")

In [0]:
source_totals = order_items.agg(
    F.sum("price").alias("source_total_price"),
    F.sum("freight_value").alias("source_total_freight")
)

fact_totals = fact_order_items.agg(
    F.sum("price").alias("fact_total_price"),
    F.sum("freight_value").alias("fact_total_freight")
)

source_totals.show()
fact_totals.show()

In [0]:
fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.fact_order_items")

In [0]:
fact_order_items.display()

# FACT PAYMENTS

In [0]:
order_payments = spark.table("lakehouse.silver.order_payments")

payment_profile = order_payments.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct(
        F.struct("order_id", "payment_sequential")
    ).alias("distinct_payment_keys"),
    F.sum(F.col("order_id").isNull().cast("int")).alias("null_order_ids"),
    F.sum(F.col("payment_sequential").isNull().cast("int")).alias("null_payment_sequential"),
    F.sum(F.col("payment_value").isNull().cast("int")).alias("null_payment_values")
)

payment_profile.show(truncate=False)

In [0]:
order_payments.printSchema()

order_payments.groupBy("payment_type").count().orderBy(
    F.desc("count")
).show(truncate=False)

In [0]:
payment_attribute_profile = order_payments.agg(
    F.sum(F.col("payment_type").isNull().cast("int")).alias("null_payment_types"),
    F.sum(F.col("payment_installments").isNull().cast("int")).alias("null_payment_installments"),
    F.sum(F.col("flag_missing_payment_type").cast("int")).alias("flagged_missing_payment_type"),
    F.sum(F.col("flag_missing_payment_installments").cast("int")).alias("flagged_missing_installments"),
    F.sum(F.col("flag_missing_payment_value").cast("int")).alias("flagged_missing_payment_value")
)

payment_attribute_profile.show(truncate=False)

In [0]:
fact_payments = (
    order_payments
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    )
)

In [0]:
payment_fact_profile = fact_payments.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct(
        F.struct("order_id", "payment_sequential")
    ).alias("distinct_payment_keys"),
    F.sum(F.col("order_id").isNull().cast("int")).alias("null_order_ids"),
    F.sum(F.col("payment_sequential").isNull().cast("int")).alias("null_payment_sequential"),
    F.sum(F.col("payment_type").isNull().cast("int")).alias("null_payment_types"),
    F.sum(F.col("payment_installments").isNull().cast("int")).alias("null_installments"),
    F.sum(F.col("payment_value").isNull().cast("int")).alias("null_payment_values")
)

payment_fact_profile.show(truncate=False)
fact_payments.printSchema()

In [0]:
# Foreign-key integrity
payment_fk_check = count_orphan_keys(
    fact_payments,
    "order_id",
    spark.table("lakehouse.gold.fact_orders"),
    "order_id"
)

print("Orphan payment order references:", payment_fk_check)

In [0]:
source_payment_total = order_payments.agg(
    F.sum("payment_value").alias("source_payment_total")
)

fact_payment_total = fact_payments.agg(
    F.sum("payment_value").alias("fact_payment_total")
)

source_payment_total.show()
fact_payment_total.show()

In [0]:
fact_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.fact_payments")

In [0]:
saved_fact_payments = spark.table("lakehouse.gold.fact_payments")

print("Persisted rows:", saved_fact_payments.count())
saved_fact_payments.printSchema()

# GOLD LAYER INTEGRATION REVIEW

In [0]:
gold_tables = [
    "dim_geography",
    "dim_customers",
    "dim_products",
    "dim_sellers",
    "dim_date",
    "fact_orders",
    "fact_order_items",
    "fact_payments"
]

for table_name in gold_tables:
    table_df = spark.table(f"lakehouse.gold.{table_name}")
    print(f"{table_name}: {table_df.count():,} rows")

Expected relationships are:
> fact_orders
- customer_sk → dim_customers.customer_sk
- customer_geography_sk → dim_geography.geography_sk
- purchase_date_sk → dim_date.date_sk
- approval_date_sk → dim_date.date_sk
- carrier_date_sk → dim_date.date_sk
- delivery_date_sk → dim_date.date_sk
- estimated_delivery_date_sk → dim_date.date_sk
  
> fact_order_items
- product_sk → dim_products.product_sk
- seller_sk → dim_sellers.seller_sk
  
> fact_payments
- order_id → fact_orders.order_id
  
  This last relationship is a fact-to-fact relationship through the shared order identifier, not a conventional dimension foreign key.
  
> dim_sellers
- geography_sk → dim_geography.geography_sk

In [0]:
fact_orders = spark.table("lakehouse.gold.fact_orders")
fact_order_items = spark.table("lakehouse.gold.fact_order_items")
fact_payments = spark.table("lakehouse.gold.fact_payments")

dim_customers = spark.table("lakehouse.gold.dim_customers")
dim_geography = spark.table("lakehouse.gold.dim_geography")
dim_products = spark.table("lakehouse.gold.dim_products")
dim_sellers = spark.table("lakehouse.gold.dim_sellers")
dim_date = spark.table("lakehouse.gold.dim_date")

gold_fk_checks = {
    "fact_orders.customer_sk": count_orphan_keys(
        fact_orders, "customer_sk", dim_customers, "customer_sk"
    ),
    "fact_orders.customer_geography_sk": count_orphan_keys(
        fact_orders, "customer_geography_sk", dim_geography, "geography_sk"
    ),
    "fact_orders.purchase_date_sk": count_orphan_keys(
        fact_orders, "purchase_date_sk", dim_date, "date_sk"
    ),
    "fact_orders.approval_date_sk": count_orphan_keys(
        fact_orders, "approval_date_sk", dim_date, "date_sk"
    ),
    "fact_orders.carrier_date_sk": count_orphan_keys(
        fact_orders, "carrier_date_sk", dim_date, "date_sk"
    ),
    "fact_orders.delivery_date_sk": count_orphan_keys(
        fact_orders, "delivery_date_sk", dim_date, "date_sk"
    ),
    "fact_orders.estimated_delivery_date_sk": count_orphan_keys(
        fact_orders, "estimated_delivery_date_sk", dim_date, "date_sk"
    ),
    "fact_order_items.product_sk": count_orphan_keys(
        fact_order_items, "product_sk", dim_products, "product_sk"
    ),
    "fact_order_items.seller_sk": count_orphan_keys(
        fact_order_items, "seller_sk", dim_sellers, "seller_sk"
    ),
    "fact_payments.order_id": count_orphan_keys(
        fact_payments, "order_id", fact_orders, "order_id"
    ),
    "dim_sellers.geography_sk": count_orphan_keys(
        dim_sellers, "geography_sk", dim_geography, "geography_sk"
    )
}

for relationship, orphan_count in gold_fk_checks.items():
    print(f"{relationship}: {orphan_count} orphan records")

In [0]:
# cardinality checks

# 1. fact_orders joined to all its dimensions
orders_joined = (
    fact_orders.alias("f")
    .join(dim_customers.alias("c"),
          F.col("f.customer_sk") == F.col("c.customer_sk"), "left")
    .join(dim_geography.alias("g"),
          F.col("f.customer_geography_sk") == F.col("g.geography_sk"), "left")
    .join(dim_date.alias("dp"),
          F.col("f.purchase_date_sk") == F.col("dp.date_sk"), "left")
    .join(dim_date.alias("da"),
          F.col("f.approval_date_sk") == F.col("da.date_sk"), "left")
    .join(dim_date.alias("dc"),
          F.col("f.carrier_date_sk") == F.col("dc.date_sk"), "left")
    .join(dim_date.alias("dd"),
          F.col("f.delivery_date_sk") == F.col("dd.date_sk"), "left")
    .join(dim_date.alias("de"),
          F.col("f.estimated_delivery_date_sk") == F.col("de.date_sk"), "left")
)

# 2. fact_order_items joined to product and seller dimensions
items_joined = (
    fact_order_items.alias("f")
    .join(dim_products.alias("p"),
          F.col("f.product_sk") == F.col("p.product_sk"), "left")
    .join(dim_sellers.alias("s"),
          F.col("f.seller_sk") == F.col("s.seller_sk"), "left")
)

# 3. fact_payments joined to fact_orders
payments_joined = (
    fact_payments.alias("p")
    .join(fact_orders.alias("o"),
          F.col("p.order_id") == F.col("o.order_id"), "left")
)

# 4. dim_sellers joined to geography
sellers_joined = (
    dim_sellers.alias("s")
    .join(dim_geography.alias("g"),
          F.col("s.geography_sk") == F.col("g.geography_sk"), "left")
)

cardinality_results = {
    "fact_orders": (fact_orders.count(), orders_joined.count()),
    "fact_order_items": (fact_order_items.count(), items_joined.count()),
    "fact_payments": (fact_payments.count(), payments_joined.count()),
    "dim_sellers": (dim_sellers.count(), sellers_joined.count())
}

for table_name, (original_count, joined_count) in cardinality_results.items():
    print(
        f"{table_name}: "
        f"original={original_count:,}, "
        f"after_join={joined_count:,}, "
        f"preserved={original_count == joined_count}"
    )

In [0]:
# Cross-fact reconciliation

order_coverage = {
    "fact_orders": fact_orders.select("order_id").distinct().count(),
    "fact_order_items": fact_order_items.select("order_id").distinct().count(),
    "fact_payments": fact_payments.select("order_id").distinct().count()
}

for table_name, order_count in order_coverage.items():
    print(f"{table_name}: {order_count:,} distinct orders")

In [0]:
item_totals = (
    fact_order_items
    .groupBy("order_id")
    .agg(
        F.sum("price").alias("item_total"),
        F.sum("freight_value").alias("freight_total")
    )
)

payment_totals = (
    fact_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_total")
    )
)

order_financial_reconciliation = (
    item_totals.alias("i")
    .join(
        payment_totals.alias("p"),
        F.col("i.order_id") == F.col("p.order_id"),
        "full_outer"
    )
    .select(
        F.coalesce(
            F.col("i.order_id"), F.col("p.order_id")
        ).alias("order_id"),
        F.col("i.item_total"),
        F.col("i.freight_total"),
        F.col("p.payment_total")
    )
    .withColumn(
        "expected_order_total",
        F.coalesce(F.col("item_total"), F.lit(0.0)) +
        F.coalesce(F.col("freight_total"), F.lit(0.0))
    )
    .withColumn(
        "difference",
        F.round(
            F.coalesce(F.col("payment_total"), F.lit(0.0)) -
            F.col("expected_order_total"),
            2
        )
    )
)

In [0]:
order_financial_reconciliation.agg(
    F.count("*").alias("orders_in_reconciliation"),
    F.sum(
        F.when(F.col("item_total").isNull(), 1).otherwise(0)
    ).alias("orders_without_items"),
    F.sum(
        F.when(F.col("payment_total").isNull(), 1).otherwise(0)
    ).alias("orders_without_payments"),
    F.sum(
        F.when(F.abs(F.col("difference")) > 0.01, 1).otherwise(0)
    ).alias("orders_with_financial_difference"),
    F.max(F.abs(F.col("difference"))).alias("maximum_absolute_difference")
).show(truncate=False)

In [0]:
order_financial_reconciliation \
    .filter(F.abs(F.col("difference")) > 0.01) \
    .orderBy(F.desc(F.abs(F.col("difference")))) \
    .show(20, truncate=False)

In [0]:
reconciliation_with_status = (
    order_financial_reconciliation.alias("r")
    .join(
        fact_orders.select("order_id", "order_status").alias("o"),
        F.col("r.order_id") == F.col("o.order_id"),
        "left"
    )
    .select(
        F.col("r.order_id"),
        F.col("o.order_status"),
        F.col("r.item_total"),
        F.col("r.freight_total"),
        F.col("r.payment_total"),
        F.col("r.difference")
    )
)

In [0]:
(
    reconciliation_with_status
    .filter(F.col("item_total").isNull())
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

In [0]:
(
    reconciliation_with_status
    .filter(F.col("payment_total").isNull())
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

In [0]:
matched_orders = reconciliation_with_status.filter(
    F.col("item_total").isNotNull() &
    F.col("payment_total").isNotNull()
)

matched_orders.agg(
    F.count("*").alias("orders_with_both_records"),
    F.sum(
        F.when(F.abs(F.col("difference")) > 0.01, 1).otherwise(0)
    ).alias("orders_with_financial_difference"),
    F.max(F.abs(F.col("difference"))).alias("maximum_absolute_difference")
).show(truncate=False)

In [0]:
(
    matched_orders
    .filter(F.abs(F.col("difference")) > 0.01)
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

In [0]:
(
    matched_orders
    .filter(F.abs(F.col("difference")) > 0.01)
    .withColumn(
        "difference_direction",
        F.when(F.col("difference") > 0, "payment_higher")
         .otherwise("payment_lower")
    )
    .groupBy("order_status", "difference_direction")
    .count()
    .orderBy("order_status", "difference_direction")
    .show(truncate=False)
)

In [0]:
item_counts = (
    fact_order_items
    .groupBy("order_id")
    .agg(F.count("*").alias("item_rows"))
)

payment_counts = (
    fact_payments
    .groupBy("order_id")
    .agg(F.count("*").alias("payment_rows"))
)

mismatch_details = (
    matched_orders
    .filter(F.abs(F.col("difference")) > 0.01)
    .join(item_counts, "order_id", "left")
    .join(payment_counts, "order_id", "left")
    .orderBy(F.desc(F.abs(F.col("difference"))))
)

mismatch_details.show(15, truncate=False)

In [0]:
(
    reconciliation_with_status
    .filter(F.col("payment_total").isNull())
    .show(truncate=False)
)

In [0]:
missing_payment_order = (
    reconciliation_with_status
    .filter(F.col("payment_total").isNull())
    .select("order_id")
)

order_payments.join(
    missing_payment_order,
    "order_id",
    "inner"
).show(truncate=False)